In [1]:
# ============================================================
# FUNCTION 8 — WEEK 5 CLEAN, MEMORY-SAFE REBUILD
# ============================================================

import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 8 data
# ------------------------------------------------------------

X = np.load("function8/initial_inputs.npy")
Y = np.load("function8/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–4 exactly once
# ------------------------------------------------------------

week1_x = np.array([[
    0.344451, 0.156410, 0.006670, 0.033817,
    0.853859, 0.614858, 0.045559, 0.106726
]])
week1_y = np.array([9.7346382479349])

week2_x = np.array([[
    0.147947, 0.193825, 0.060981, 0.008004,
    0.719306, 0.453461, 0.045592, 0.938970
]])
week2_y = np.array([9.894432442301])

week3_x = np.array([[
    0.129113, 0.207318, 0.165052, 0.244138,
    0.603646, 0.535759, 0.168630, 0.681517
]])
week3_y = np.array([9.9592828135141])

week4_x = np.array([[
    0.078500, 0.107318, 0.212619, 0.144138,
    0.703646, 0.485325, 0.134507, 0.581517
]])
week4_y = np.array([9.9632714558391])

X = np.vstack([
    X,
    week1_x,
    week2_x,
    week3_x,
    week4_x
])

Y = np.concatenate([
    Y,
    week1_y,
    week2_y,
    week3_y,
    week4_y
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nBest observed input:", best_x)
print("Best observed output:", best_y)

# ------------------------------------------------------------
# 3. Fit an ARD Gaussian Process
# ------------------------------------------------------------

kernel = (
    C(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.full(8, 0.5),
        length_scale_bounds=(0.02, 4.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=7,
    random_state=58
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)

# ------------------------------------------------------------
# 4. Inspect modelled input sensitivity
# ------------------------------------------------------------

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nFitted lengthscales:", lengthscales)
print("Normalised input influence:")

for i, value in enumerate(importance, start=1):
    print(f"x{i}: {value:.4f}")

# ------------------------------------------------------------
# 5. Generate candidates
#
# Most candidates refine the Week 4 best.
# Some search between Weeks 3 and 4 because both were strong.
# A smaller global pool preserves exploration in eight
# dimensions.
# ------------------------------------------------------------

rng = np.random.default_rng(58)

very_local = rng.normal(
    loc=best_x,
    scale=[
        0.012, 0.015, 0.015, 0.015,
        0.018, 0.018, 0.015, 0.020
    ],
    size=(6_000, 8)
)

local = rng.normal(
    loc=best_x,
    scale=[
        0.035, 0.045, 0.045, 0.045,
        0.050, 0.050, 0.040, 0.055
    ],
    size=(8_000, 8)
)

wider_local = rng.normal(
    loc=best_x,
    scale=[
        0.070, 0.085, 0.085, 0.085,
        0.090, 0.090, 0.080, 0.100
    ],
    size=(4_000, 8)
)

# Candidates between the two strongest observations
blend_weights = rng.uniform(
    0,
    1,
    size=(4_000, 1)
)

blended_candidates = (
    blend_weights * week3_x
    + (1 - blend_weights) * week4_x
)

blended_candidates += rng.normal(
    loc=0,
    scale=[
        0.018, 0.022, 0.022, 0.022,
        0.025, 0.025, 0.020, 0.028
    ],
    size=(4_000, 8)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(2_000, 8)
)

candidates = np.vstack([
    very_local,
    local,
    wider_local,
    blended_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("\nCandidates before filtering:", len(candidates))

# ------------------------------------------------------------
# 6. Remove almost-duplicate candidates
# ------------------------------------------------------------

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    minimum_distance > 0.006
]

print("Candidates after filtering:", len(candidates))

# ------------------------------------------------------------
# 7. Predict in memory-safe batches
# ------------------------------------------------------------

def predict_in_batches(model, points, batch_size=1500):

    means = []
    standard_deviations = []

    for start in range(0, len(points), batch_size):

        batch = points[
            start:start + batch_size
        ]

        batch_mean, batch_std = model.predict(
            batch,
            return_std=True
        )

        means.append(batch_mean)
        standard_deviations.append(batch_std)

    return (
        np.concatenate(means),
        np.concatenate(standard_deviations)
    )

mean, std = predict_in_batches(
    gp,
    candidates
)

# ------------------------------------------------------------
# 8. Expected Improvement
# ------------------------------------------------------------

xi = 0.002

improvement = mean - best_y - xi

with np.errstate(divide="ignore", invalid="ignore"):

    z = improvement / std

    expected_improvement = (
        improvement * norm.cdf(z)
        + std * norm.pdf(z)
    )

expected_improvement[std < 1e-12] = 0

# ------------------------------------------------------------
# 9. Moderate UCB exploration
# ------------------------------------------------------------

kappa = 0.55
ucb = mean + kappa * std

# ------------------------------------------------------------
# 10. Retain competitive candidates
# ------------------------------------------------------------

mean_filter = mean >= (best_y - 0.08)

if mean_filter.sum() < 100:
    mean_filter = mean >= np.percentile(mean, 90)

filtered_candidates = candidates[mean_filter]
filtered_mean = mean[mean_filter]
filtered_std = std[mean_filter]
filtered_ei = expected_improvement[mean_filter]
filtered_ucb = ucb[mean_filter]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

# ------------------------------------------------------------
# 11. Combine EI and UCB
# ------------------------------------------------------------

ei_normalised = (
    filtered_ei - filtered_ei.min()
) / (
    np.ptp(filtered_ei) + 1e-12
)

ucb_normalised = (
    filtered_ucb - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb) + 1e-12
)

# Strong exploitation because Weeks 3 and 4 established
# a consistently strong region
acquisition = (
    0.82 * ei_normalised
    + 0.18 * ucb_normalised
)

chosen_index = np.argmax(acquisition)
week5_query = filtered_candidates[chosen_index]

# ------------------------------------------------------------
# 12. Print Week 5 proposal
# ------------------------------------------------------------

print("\nMethod:")
print("Local filtered Expected Improvement with small UCB component")

print("\nSuggested Week 5 query:")
print(week5_query)

print("\nPortal format:")
print("-".join(
    f"{value:.6f}"
    for value in week5_query
))

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted standard deviation:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "UCB:",
    filtered_ucb[chosen_index]
)

print(
    "Distance from Week 4:",
    np.linalg.norm(
        week5_query - week4_x[0]
    )
)

print(
    "Distance from Week 3:",
    np.linalg.norm(
        week5_query - week3_x[0]
    )
)

X shape: (44, 8)
Y shape: (44,)

Best observed input: [0.0785   0.107318 0.212619 0.144138 0.703646 0.485325 0.134507 0.581517]
Best observed output: 9.9632714558391

Fitted kernel:
1.65**2 * Matern(length_scale=[1.91, 2.81, 1.48, 4, 4, 4, 1.97, 4], nu=2.5) + WhiteKernel(noise_level=1e-10)

Fitted lengthscales: [1.91334652 2.80776349 1.48344133 4.         4.         4.
 1.96742248 4.        ]
Normalised input influence:
x1: 0.1707
x2: 0.1163
x3: 0.2202
x4: 0.0817
x5: 0.0817
x6: 0.0817
x7: 0.1660
x8: 0.0817

Candidates before filtering: 24000
Candidates after filtering: 24000
Candidates passing mean filter: 20351

Method:
Local filtered Expected Improvement with small UCB component

Suggested Week 5 query:
[0.09400026 0.17677621 0.10222611 0.14055512 0.98380915 0.56636946
 0.15795635 0.56803457]

Portal format:
0.094000-0.176776-0.102226-0.140555-0.983809-0.566369-0.157956-0.568035

Predicted mean: 10.045038776620608
Predicted standard deviation: 0.09503496097210011
Expected Improvement

/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 4.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 4.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 4.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/co